In [ ]:
import polars as pl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import datetime
import os
import sys
from scipy.stats import spearmanr

In [ ]:
# [자동 경로 설정] 프로젝트 루트(/Users/yumi/Seven-eleven)를 정확히 찾아 데이터를 연결합니다.
def get_root():
    # 현재 노트북 위치: Seven-eleven/eda/ipynb/yumi/eda.ipynb
    # 위로 3단계를 올라가면 루트입니다.
    path = os.path.abspath(os.getcwd())
    for _ in range(10): # 최대 10단계까지 상위 디렉토리 탐색
        if os.path.exists(os.path.join(path, 'data')):
            return path
        parent = os.path.dirname(path)
        if parent == path: break
        path = parent
    return os.getcwd() # 실패 시 현재 디렉토리 반환

ROOT = get_root()
B2_PATH = os.path.join(ROOT, 'data/processed/B2_POS_SALE.parquet')
B4_PATH = os.path.join(ROOT, 'data/processed/B4_CLEAN_FOOD_ITEM.parquet')
B4_RAW_PATH = os.path.join(ROOT, 'data/raw/B4_ITEM_DV_INFO.csv')

print(f"✅ 프로젝트 루트 확인: {ROOT}")
print(f"📂 데이터 경로:
- B2: {B2_PATH}
- B4: {B4_PATH}")

In [ ]:
# 🚀 [데이터 로드 및 결합]
print("📊 데이터를 로드하는 중...")

if os.path.exists(B2_PATH):
    b2_lazy = pl.scan_parquet(B2_PATH)
    
    # B4 (마스터) 로드
    if os.path.exists(B4_PATH):
        b4_df = pl.read_parquet(B4_PATH)
    else:
        print("⚠️ 가공된 B4 파일이 없어 원본 CSV를 로드합니다.")
        b4_df = pl.read_csv(B4_RAW_PATH, schema_overrides={"ITEM_CD": pl.String})

    # 1. 판매 데이터를 상품별로 요약
    df_item_sales = (
        b2_lazy.group_by("상품코드")
        .agg([
            pl.col("판매금액").sum().alias("총매출액"),
            pl.col("판매수량").sum().alias("총판매량")
        ])
    )

    # 2. 마스터 데이터 결합 (타입 일치 처리)
    # B4의 상품코드 컬럼명이 ITEM_CD인 경우 처리
    if "ITEM_CD" in b4_df.columns:
        b4_df = b4_df.rename({"ITEM_CD": "상품코드"})
    
    b4_df = b4_df.with_columns(pl.col("상품코드").cast(pl.Utf8))

    df_merged = (
        df_item_sales.join(
            b4_df.lazy(), 
            on="상품코드", 
            how="inner"
        )
    ).collect() 

    print(f"✅ 결합 완료! 분석 대상 상품 수: {len(df_merged):,}개")
else:
    print(f"❌ 판매 데이터 파일을 찾을 수 없습니다: {B2_PATH}")

In [ ]:
# 🚀 [4. 중분류별 매출 80% 기여 핵심 상품 분석]
if 'df_merged' in globals():
    print("📊 중분류별 매출 80% 기여 상품 분석 시작...")
    
    results = []
    middle_categories = df_merged["ITEM_MDDV_NM"].unique().to_list()

    for cat in middle_categories:
        if cat is None: continue
        
        # 해당 중분류 필터링 및 매출 순 정렬
        df_cat = df_merged.filter(pl.col("ITEM_MDDV_NM") == cat).sort("총매출액", descending=True)
        
        total_items = len(df_cat)
        if total_items == 0: continue
        
        total_sales = df_cat["총매출액"].sum()
        if total_sales == 0: continue
        
        # 누적 비중 계산
        df_cat = df_cat.with_columns([
            (pl.col("총매출액").cum_sum() / total_sales * 100).alias("누적비중")
        ])
        
        # 매출 80% 안에 드는 상품들 필터링
        df_top80 = df_cat.filter(pl.col("누적비중") <= 80.5)
        if len(df_top80) == 0: df_top80 = df_cat.head(1)
            
        top80_item_names = df_top80["ITEM_NM"].to_list()
        top80_count = len(top80_item_names)
        
        results.append({
            "중분류명": cat,
            "전체상품수": total_items,
            "상위80%상품수": top80_count,
            "상위80%상품비중(%)": round((top80_count / total_items * 100), 2),
            "매출80%기여_상품리스트": ", ".join(top80_item_names),
            "카테고리총매출": total_sales
        })

    # 결과 데이터프레임 생성
    df_final_80 = pl.DataFrame(results).sort("카테고리총매출", descending=True)
    
    # 엑셀 저장
    output_path = os.path.join(ROOT, 'eda/ipynb/yumi/category_top80_analysis.xlsx')
    df_final_80.to_pandas().to_excel(output_path, index=False)

    print(f"✅ 분석 완료! 결과가 엑셀로 저장되었습니다: {output_path}")
    print("\n🏆 [매출 상위 10개 중분류의 80% 핵심 상품 현황]")
    display(df_final_80.select(["중분류명", "전체상품수", "상위80%상품수", "상위80%상품비중(%)"]).head(10))
else:
    print("❌ 'df_merged' 변수를 찾을 수 없습니다. 위 셀들을 먼저 실행해 주세요.")